# 02 - Style baseline: TF-IDF + logistic regression

This notebook reproduces the ISOT baseline and shows why a high accuracy on
this dataset does not mean the model checks facts.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SCRIPTS = PROJECT_ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))
import config
import baseline_isot as bi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

dataset, load_stats = bi.load_isot()
rows_before = len(dataset)
dataset = dataset.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
dataset["text_as_is"] = dataset["text"]
dataset["text_stripped"] = dataset["text"].apply(bi.strip_source_tag)
train_index, test_index = train_test_split(
    dataset.index, test_size=config.BASELINE_TEST_SIZE,
    stratify=dataset["label"], random_state=config.BASELINE_RANDOM_STATE)
print("rows:", len(dataset), "train:", len(train_index), "test:", len(test_index))

INFO loaded 22851 rows from Fake.csv (630 empty bodies dropped)


INFO loaded 21416 rows from True.csv (1 empty bodies dropped)


INFO ISOT total=44267 fake=22851 true=21416


rows: 38639 train: 30911 test: 7728


## Train one variant

A small helper trains TF-IDF + logistic regression and returns the accuracy and
confusion matrix.

In [2]:
def run_variant(text_col):
    vec = TfidfVectorizer(max_features=config.BASELINE_TFIDF_MAX_FEATURES,
                          stop_words=config.BASELINE_TFIDF_STOP_WORDS)
    Xtr = vec.fit_transform(dataset.loc[train_index, text_col])
    Xte = vec.transform(dataset.loc[test_index, text_col])
    clf = LogisticRegression(max_iter=config.BASELINE_LOGREG_MAX_ITER,
                             random_state=config.BASELINE_RANDOM_STATE)
    clf.fit(Xtr, dataset.loc[train_index, "label"])
    pred = clf.predict(Xte)
    y = dataset.loc[test_index, "label"]
    acc = accuracy_score(y, pred)
    cm = confusion_matrix(y, pred, labels=[0, 1])
    return acc, cm, vec, clf, pred

acc_as, cm_as, vec_as, clf_as, pred_as = run_variant("text_as_is")
acc_st, cm_st, vec_st, clf_st, pred_st = run_variant("text_stripped")
print("as-is accuracy:", round(acc_as, 4))
print("stripped accuracy:", round(acc_st, 4))

as-is accuracy: 0.9802
stripped accuracy: 0.9744


## Results table and confusion matrices

In [3]:
print("variant   accuracy")
print("as-is    ", round(acc_as, 4))
print("stripped ", round(acc_st, 4))
print("as-is confusion matrix (rows actual, cols predicted):")
print(cm_as)
print("stripped confusion matrix:")
print(cm_st)

variant   accuracy
as-is     0.9802
stripped  0.9744
as-is confusion matrix (rows actual, cols predicted):
[[3374  116]
 [  37 4201]]
stripped confusion matrix:
[[3340  150]
 [  48 4190]]


## Most influential words

The logistic-regression coefficients tell us which words push the prediction
toward Fake (negative) or True (positive). The agency name dominates.

In [4]:
feature_names = vec_as.get_feature_names_out()
coefs = clf_as.coef_[0]
order = np.argsort(coefs)
print("Top words toward Fake (most negative coefficients):")
for i in order[:15]:
    print(f"  {feature_names[i]:20s} {coefs[i]:.3f}")
print("Top words toward True (most positive coefficients):")
for i in order[-15:][::-1]:
    print(f"  {feature_names[i]:20s} {coefs[i]:.3f}")

Top words toward Fake (most negative coefficients):
  just                 -6.798
  image                -6.742
  featured             -6.629
  read                 -5.858
  gop                  -5.525
  com                  -5.032
  getty                -4.766
  hillary              -4.543
  mr                   -4.501
  like                 -4.358
  watch                -4.191
  america              -4.118
  pic                  -4.090
  images               -4.015
  obama                -3.602
Top words toward True (most positive coefficients):
  reuters              23.141
  said                 19.162
  washington           6.795
  wednesday            5.474
  tuesday              5.379
  thursday             4.854
  republican           4.787
  friday               4.475
  monday               4.446
  minister             4.062
  nov                  3.970
  presidential         3.652
  comment              3.331
  spokesman            3.254
  democratic           3.211


## What the baseline really measures

The model is not checking facts. It is learning writing style and the wire-service
dateline. Removing the dateline drops accuracy only slightly (0.980 to 0.974),
which shows the style signal is strong on its own.

In [5]:
print("notebook 02 complete")

notebook 02 complete
